# Extract Arctic subset from ACCESS-OM2 MLT budget files

Subsets the full-global ACCESS-OM2 0.25° IAF cycle 6 MLT budget data to **north of 60°N**
and writes the result to a new directory on scratch, preserving the original directory
structure so the output can be used with the same notebook code.

Files written:
- `output{NNN}/ocean/ocean_daily.nc` — daily SST + MLD, one file per year
- `post_processed_diags/om2_025_MLT_clim.nc` — daily climatology
- `post_processed_diags/om2_025_MLT_thresh.nc` — 90th-percentile threshold
- `post_processed_diags/mlt_budget_online_stavg/mlt_budget_stavg_daily_online_output{NNN}.nc` — daily budget, one file per year
- `post_processed_diags/mlt_budget_online_stavg/mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc` — budget climatology

**Note:** This notebook reads and writes large files sequentially. On Gadi it is best
run from an ARE JupyterLab session with at least 32 GB RAM, or submitted as a PBS job
using `papermill`:
```bash
papermill 03_extract_arctic_subset.ipynb 03_extract_arctic_subset_out.ipynb
```

In [ ]:
import xarray as xr
from pathlib import Path
import time

In [ ]:
from dask.distributed import Client
client = Client()
client

## Configuration

In [ ]:
# ── Input data ────────────────────────────────────────────────────────────────
base = Path('/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/')

START_OUTPUT = 357   # 2010
END_OUTPUT   = 366   # 2019
outputs = list(range(START_OUTPUT, END_OUTPUT + 1))

# ── Output ────────────────────────────────────────────────────────────────────
out_base = Path('/scratch/m35/nm5072/arctic_mhw_subset/')

# ── Spatial subset ────────────────────────────────────────────────────────────
LAT_MIN = 60.0   # degrees N — north of this latitude only

# ── Variables to keep per file type ──────────────────────────────────────────
MLT_VARS    = ['temp_in_mld', 'mld']
BUDGET_VARS = ['mlt_tendency', 'advection', 'vert_mixing',
               'entrainment', 'surface_flux', 'sw_pen', 'residual']
CLIM_VARS   = ['temp']

# ── NetCDF output compression ─────────────────────────────────────────────────
COMPRESS = {'zlib': True, 'complevel': 4}

print(f"Input:  {base}")
print(f"Output: {out_base}")
print(f"Subset: yt_ocean >= {LAT_MIN}°N")
print(f"Years:  {outputs[0]} ({1958 + outputs[0] - 305}) – {outputs[-1]} ({1958 + outputs[-1] - 305})")

## Helper function

In [ ]:
def extract_and_save(in_path, out_path, vars_keep, open_chunks, lat_min=LAT_MIN):
    """
    Open in_path (lazy), subset to yt_ocean >= lat_min, keep vars_keep,
    write compressed NetCDF to out_path.  Skips if out_path already exists.
    """
    in_path  = Path(in_path)
    out_path = Path(out_path)

    if out_path.exists():
        print(f"    SKIP — already exists: {out_path.name}")
        return

    out_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    ds = xr.open_dataset(in_path, chunks=open_chunks, decode_timedelta=False)
    ds_sub = ds[vars_keep].sel(yt_ocean=slice(lat_min, None))
    encoding = {v: COMPRESS for v in ds_sub.data_vars}
    ds_sub.to_netcdf(out_path, encoding=encoding)
    ds.close()

    size_mb = out_path.stat().st_size / 1e6
    print(f"    {out_path.name}: {size_mb:.0f} MB in {time.time() - t0:.0f} s")

## 1. Daily SST files

In [ ]:
print("=== MLT daily files ===")
for o in outputs:
    year = 1958 + (o - 305)
    in_f  = base / f'output{o:03d}/ocean/ocean_daily.nc'
    out_f = out_base / f'output{o:03d}/ocean/ocean_daily.nc'
    print(f"  {year}  (output{o:03d})")
    extract_and_save(in_f, out_f, MLT_VARS, open_chunks={'time': 30})

print("\nMLT done.")

## 2. Daily budget files

In [ ]:
print("=== Budget daily files ===")
for o in outputs:
    year  = 1958 + (o - 305)
    fname = f'mlt_budget_stavg_daily_online_output{o:03d}.nc'
    in_f  = base / f'post_processed_diags/mlt_budget_online_stavg/{fname}'
    out_f = out_base / f'post_processed_diags/mlt_budget_online_stavg/{fname}'
    print(f"  {year}  (output{o:03d})")
    extract_and_save(in_f, out_f, BUDGET_VARS, open_chunks={'time': 10})

print("\nBudget done.")

## 3. Static files — climatology, threshold, budget climatology

In [ ]:
# (dayofyear dimension, not time — use spatial chunks)
CLIM_CHUNKS   = {'dayofyear': 30, 'yt_ocean': 216, 'xt_ocean': 240}
# budget clim has a time dimension (12 monthly values) — small file, minimal chunking
BCLIM_CHUNKS  = {'time': 12, 'yt_ocean': 216, 'xt_ocean': 240}

static_files = [
    (
        base / 'post_processed_diags/om2_025_MLT_clim.nc',
        out_base / 'post_processed_diags/om2_025_MLT_clim.nc',
        CLIM_VARS, CLIM_CHUNKS,
    ),
    (
        base / 'post_processed_diags/om2_025_MLT_thresh.nc',
        out_base / 'post_processed_diags/om2_025_MLT_thresh.nc',
        CLIM_VARS, CLIM_CHUNKS,
    ),
    (
        base / 'post_processed_diags/mlt_budget_online_stavg/'
               'mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc',
        out_base / 'post_processed_diags/mlt_budget_online_stavg/'
                   'mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc',
        BUDGET_VARS, BCLIM_CHUNKS,
    ),
]

print("=== Static files ===")
for in_f, out_f, vars_keep, chunks in static_files:
    print(f"  {in_f.name}")
    extract_and_save(in_f, out_f, vars_keep, open_chunks=chunks)

print("\nStatic files done.")

## 4. Verify output

In [ ]:
import subprocess

result = subprocess.run(['du', '-sh', str(out_base)], capture_output=True, text=True)
print(f"Total size: {result.stdout.strip()}")

print("\nDirectory structure:")
for p in sorted(out_base.rglob('*.nc')):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.relative_to(out_base)}  ({size_mb:.0f} MB)")

In [ ]:
# Quick sanity check — open one output MLT file and print lat range and variable info
ds_check = xr.open_dataset(out_base / f'output{END_OUTPUT:03d}/ocean/ocean_daily.nc')
print(ds_check)
print(f"\nyt_ocean range: {float(ds_check.yt_ocean.min()):.2f} — {float(ds_check.yt_ocean.max()):.2f} °N")
ds_check.close()